# Formatting Anki Flashcards

## Goal

While using Anki over the years, I struggled with following some common best practices―like keeping each card atomic, choosing the best direction to test, avoiding topic-labels, having clear cues, etc. Building great cards takes time which, in my experience, it's a major friction to creating new cards and reviewing them. 

Throughout this notebook, we want to iteratively build a good way to prompt a small language model, following basic best practices. We will start with a simple LLM call and systematically build a more robust and performant solution. 

We will follow this progression:
1. Simple prompt
1. Simple prompt + Structured Output
1. Simple prompt + Structured Output + Thinking
1. Improve prompt based on error analysis and frequent failure modes

## Setup environment

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from anki.collection import Collection

from addon.application.use_cases.note_counter import is_note_marked_for_review
from addon.infrastructure.configuration.settings import AddonConfig

In [3]:
# Open an existing collection
col = Collection("/home/gianluca/.local/share/Anki2/User 1/collection.anki2")

# Do something with the collection
print(f"Number of notes: {col.note_count()}")
print(f"Number of cards: {col.card_count()}")

Number of notes: 3194
Number of cards: 3256


### Connect to Inference Server

In [4]:
from IPython.display import Markdown, display

from addon.domain.entities.note import AddonNote
from addon.infrastructure.llm.schemas import AddonNoteChanges


def display_addon_note(addon_note: AddonNote | AddonNoteChanges | str) -> None:
    if isinstance(addon_note, str):
        print(addon_note)
    elif isinstance(addon_note, (AddonNote, AddonNoteChanges)):
        display(
            Markdown(
                f"**Front:** {addon_note.front}\n\n"
                f"**Back:** {addon_note.back}\n\n"
                # f"**Tags:** {addon_note.tags}"
            )
        )
    else:
        raise ValueError(
            f"Expected str, AddonNote, or AddonNoteChanges, found {type(addon_note)}"
        )

In [5]:
from addon.infrastructure.external_services.openai import OpenAIClient


def answer(input: str | list[dict], guided_json=None, **kwargs):
    """Helper function to prompt LLM.

    input: Either a string (completions API) or list of message dicts (chat completion)
    kwargs: Can override config values like max_tokens, temperature, etc.
            extra_body: dict whose contents are merged into the request payload.
    """
    extra_body = kwargs.pop("extra_body", {})
    config = AddonConfig.create_nullable(kwargs)
    client = OpenAIClient.create(config)

    run_kwargs = {**extra_body}
    if guided_json is not None:
        run_kwargs["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": "addon_note_changes",
                "schema": guided_json,
            },
        }

    content = client.run(input, **run_kwargs)
    reasoning_content = client.last_reasoning_content
    cleaned_content = content.replace("<think>\n\n</think>\n\n", "")

    if reasoning_content:
        print("=== Thinking ===")
        print(reasoning_content)
        print("=== End Thinking ===\n")

    display_addon_note(cleaned_content)

    if guided_json is not None:
        suggested_changes = AddonNoteChanges.model_validate_json(
            cleaned_content
        )
        return (suggested_changes, reasoning_content)

    return (cleaned_content, reasoning_content)

Now we have everything we need to tell the LLM to make some changes to our Anki flashcards.

Let's pull a few notes currently marked for review.

In [6]:
deck_id = col.decks.current()["id"]
query = f"did:{deck_id}"
note_ids = col.find_notes(query)

NUM_NOTES_NEEDED = 3

flagged_notes = []
for note_id in note_ids:
    if is_note_marked_for_review(col, note_id):
        note = col.get_note(note_id)
        if "personal" not in note.tags:
            flagged_notes.append(note)
            if len(flagged_notes) >= NUM_NOTES_NEEDED:
                break

In [7]:
print(f"Number of flagged notes: {len(flagged_notes)}")

Number of flagged notes: 3


## Show original version of flagged notes

In [8]:
from addon.application.services.formatter_service import AnkiNoteAdapter

addon_notes = [AnkiNoteAdapter.to_addon_note(note) for note in flagged_notes]

for i, note in enumerate(addon_notes):
    print(f"--- Note {i + 1} ---")
    display_addon_note(note)
    print()

--- Note 1 ---


**Front:** How can we compute the length of a vector $\overrightarrow{v} = \begin{bmatrix} v_{1} \\ v_{2} \end{bmatrix}$?

**Back:** $\left\|\mathbf{v}\right\| = \sqrt{v_1^2 + v_2^2}$




--- Note 2 ---


**Front:** L2-norm: also known as

**Back:** Euclidian distance




--- Note 3 ---


**Front:** Name for _bagging_ in Statistics

**Back:** Bootstrapping



## Simple Prompt

**⚠️ WARNING:** This section is for experimentation and baseline comparison only. We will not use such an approach in production. 

Let's investigate what a simple prompt could achieve. This approach is clearly flawed. Without constraining the LLM's output, we have no guarantees of a consistent format (e.g., that all fields are present, that it is valid JSON, etc.). Later in the notebook, we will use "structured output" to overcome these issues. 

Let's build a simple prompt that instructs the LLM to improve Anki flashcards based on some basic guidelines.

In [9]:
simple_prompt_template = (
    "Look at this flashcard. How would you improve it? "
    "Keep in mind that flashcards should be atomic, concise, and accurate. "
    "Do not explain the changes.\n\nFront: {note.front}\nBack: {note.back}"
)

# Print a sample prompt
print(simple_prompt_template.format(note=addon_notes[0]))

Look at this flashcard. How would you improve it? Keep in mind that flashcards should be atomic, concise, and accurate. Do not explain the changes.

Front: How can we compute the length of a vector $\overrightarrow{v} = \begin{bmatrix} v_{1} \\ v_{2} \end{bmatrix}$?
Back: $\left\|\mathbf{v}\right\| = \sqrt{v_1^2 + v_2^2}$


In [10]:
for i, addon_note in enumerate(addon_notes):
    print(f"--- Note {i + 1} ---")
    answer(
        input=[
            {
                "role": "user",
                "content": simple_prompt_template.format(note=addon_note),
            }
        ],
        max_tokens=1_000,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    print()

--- Note 1 ---
Front: What is the formula for the magnitude of vector $\overrightarrow{v} = \begin{bmatrix} v_{1} \\ v_{2} \end{bmatrix}$?
Back: $\sqrt{v_1^2 + v_2^2}$

--- Note 2 ---
Front: What is another name for the L2-norm?
Back: Euclidean norm

--- Note 3 ---
Front: What does bootstrapping approximate in statistics?
Back: Bagging



Despite the very simple prompt, the model is doing a good job. However, we probably need structured output/constrained decoding to ensure the LLM respects a predefined format. This will make extracting the suggestions easier.

## Simple Prompt + Structured Output

Let's reuse the `pydantic` schema we have already defined for the main project.

In [11]:
from addon.infrastructure.llm.schemas import AddonNoteChanges

for i, addon_note in enumerate(addon_notes):
    print(f"--- Note {i + 1} ---")
    content, _ = answer(
        input=[
            {
                "role": "user",
                "content": simple_prompt_template.format(note=addon_note),
            }
        ],
        guided_json=AddonNoteChanges.model_json_schema(),
        max_tokens=2_000,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    display_addon_note(content)
    print()

--- Note 1 ---
{
  "front": "What is the formula for the magnitude of a vector $\\mathbf{v} = \\begin{bmatrix} v_1 \\\\ v_2 \\end{bmatrix}$?",
  "back": "$\\|\\mathbf{v}\\| = \\sqrt{v_1^2 + v_2^2}$"
}


**Front:** What is the formula for the magnitude of a vector $\mathbf{v} = \begin{bmatrix} v_1 \\ v_2 \end{bmatrix}$?

**Back:** $\|\mathbf{v}\| = \sqrt{v_1^2 + v_2^2}$




--- Note 2 ---
{
  "front": "Alternative name for the L2-norm",
  "back": "Euclidean norm"
}


**Front:** Alternative name for the L2-norm

**Back:** Euclidean norm




--- Note 3 ---
{
  "front": "What is the statistical term for resampling with replacement?",
  "back": "Bootstrapping"
}


**Front:** What is the statistical term for resampling with replacement?

**Back:** Bootstrapping



## Reasoning + Structured Output + Old Prompt

Let's compare it with the current version we have in the `main` branch.

In [12]:
from addon.application.services.formatter_service import NoteFormatter

config = AddonConfig.create_nullable(
    {
        "mode": "v1/chat/completions",
        "url": "http://iamgianluca.ddns.net:8080/v1/completions",
        "max_tokens": 2_000,  # needs headroom for reasoning + structured output
    }
)
for k, v in config.__dict__.items():
    if k == "url":  # hide inference server url
        continue
    print(f"{k}: {v}")

model_name: ./Qwen3.5-27B-UD-Q4_K_XL.gguf
temperature: 1.0
max_tokens: 2000
top_p: 0.95
top_k: 20
min_p: 0.0


In [13]:
openai = OpenAIClient.create(config)
formatter = NoteFormatter(openai)

for i, addon_note in enumerate(addon_notes):
    print(f"--- Note {i + 1} ---")
    new_note = formatter.format(note=addon_note)
    display_addon_note(new_note)
    print()

--- Note 1 ---


**Front:** In linear algebra, how do you compute the length of a vector $\overrightarrow{v} = \begin{bmatrix} v_{1} \\ v_{2} \end{bmatrix}$?

**Back:** $\left\|\mathbf{v}\right\| = \sqrt{v_1^2 + v_2^2}$




--- Note 2 ---


**Front:** In mathematics, what is the L2-norm also known as?

**Back:** Euclidean distance




--- Note 3 ---


**Front:** In ML, what is the statistical term for bagging?

**Back:** Bootstrapping



After cleaning up the prompt — removing the redundant inline JSON schema, updating examples to the `In <domain>, <question>?` format, and dropping `tags` from the output — the results are much closer to the target format. The detailed formatting rules and few-shot examples give the model clear guidance, and reasoning helps it apply them correctly.

## Reasoning + Structured Output + Simple Prompt

Let's try to let the LLM reason, while still perform constrained decoding.

In [14]:
from addon.infrastructure.llm.schemas import AddonNoteChanges

for i, addon_note in enumerate(addon_notes):
    print(f"--- Note {i + 1} ---")
    content, _ = answer(
        input=[
            {
                "role": "user",
                "content": simple_prompt_template.format(note=addon_note),
            }
        ],
        guided_json=AddonNoteChanges.model_json_schema(),
        max_tokens=2_000,
        extra_body={"chat_template_kwargs": {"enable_thinking": True}},
    )
    display_addon_note(content)
    print()

--- Note 1 ---
{
  "front": "What is the formula for the magnitude of a 2D vector $(v_1, v_2)$?",
  "back": "$\\sqrt{v_1^2 + v_2^2}$"
}


**Front:** What is the formula for the magnitude of a 2D vector $(v_1, v_2)$?

**Back:** $\sqrt{v_1^2 + v_2^2}$




--- Note 2 ---
{
  "front": "What is the L2-norm also known as?",
  "back": "Euclidean distance"
}


**Front:** What is the L2-norm also known as?

**Back:** Euclidean distance




--- Note 3 ---
{
  "front": "What is the term for _bagging_ in statistics?",
  "back": "Bootstrapping"
}


**Front:** What is the term for _bagging_ in statistics?

**Back:** Bootstrapping



In [15]:
col.close()

## TODO

- [x] Simple prompt
- [x] Simple prompt + Constrained Decoding
- [x] Simple prompt + Constrained Decoding + Reasoning
- [ ] Simple prompt + Chain of Thought
- [x] Cleaner representation of existing addon note in the prompt passed to the LLM
- [x] Pass note to change as `user` instead of `system` role
- [ ] Two-step process (critique → refine)
- [ ] Agent
- [ ] Check if we have a class that we can use to read the collection from the hard disk and convert it to `AddonNote` instead of `Note`. In that way we can operate with domain objects instead of external dependencies. The same class should be used to convert the `AddonNote` back to `Note` (so maybe we should keep track of the `note_id`
- [ ] Once we have that class, we should update the rest of the codebase accordingly (e.g., note formatter, note counter, etc.)
- [ ] ...